# Notebook #2: HCS Plate Exploration

This notebook is used to explore data from an HCS-SMLM experiment. It proposes to explore it at several levels:
* a single well,
* between wells of the same protein (protein replicates),
* between different proteins of the same plate, 
* multi-plates comparisons...

And to look at all the photophysical parameters : intensity per localization, intensity per molecule, ON and OFF times, number of blinks...

At the end of the notebook, it is **possible to export a CSV file** containing values use to perform statistical tests in another software (R studio, GraphPad...).

## Prerequisites
Normally, you should run the first notebook before using this one. If you haven't, please refer to the installation guide for the virtual environment, and more generally to the first notebook, before going any further in this notebook.

### Loading librairies and HCS Experiment

In the following section, we will load Python librairies required as well as the HCS experiment. Run the next cell to load librairies and dependencies:

In [1]:
# Run to load librairies
%reload_ext autoreload
%autoreload 2

import os
import sys

import ipywidgets as widgets

from ipyfilechooser import FileChooser
from IPython.display import display, clear_output

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.io_utils import create_database
from src.visualization.count_total_locs_and_clusters import count_total_locs_and_clusters
from src.visualization.heatmap_single_parameters import plot_plate_heatmap
from src.visualization.boxplots_same_protein import plot_well_boxplots
from src.visualization.boxplot_same_well import plot_fov_boxplots
from src.save.save_as_excel import process_and_export_plates


print("Librairies and dependencies correctly loaded!")

Librairies and dependencies correctly loaded!


Now, we will load HCS Experiment. You will need to load only the directory containing the whole plate.

> ⚠️ The code is not made to work with individual wells. ⚠️

*By default, the HCS plugin used generates a main directory named W1. Thus, the code is adapted to get directory with W1. The naming can be changed in the code.*

In [2]:
# Load directory

fc = FileChooser(os.getcwd())
fc.show_only_dirs = True
fc.title = '<b>Choose the directory (must contains "W1"):</b>'

def check_repertory(chooser):
    clear_output(wait=True)
    display(fc)
    
    selected_dir = chooser.selected_path
    folder_name = os.path.basename(selected_dir)
    
    if "W1" in folder_name.upper():
        print(f"Pathway : {selected_dir}")
    else:
        print(f"Error : Repertory ('{folder_name}') doesn't have 'W1' in its name. Please choose another one.")

fc.register_callback(check_repertory)
display(fc)

FileChooser(path='/Users/aneuhaus/Desktop/asha/asha/notebooks', filename='', title='<b>Choose the directory (m…

In [21]:
# Check pathway loaded
pathway = fc.selected_path
if pathway and "W1" in os.path.basename(pathway).upper():
    print(f"Ready to analyze : {pathway}")
else:
    print("Please choose a correct pathway.")

Ready to analyze : /Users/aneuhaus/Desktop/asha/DATA/W1_bis


### Explore one HCS plate data

In this section, various visualization methods can be used to explore the data at different scales. For a comparison including multiple plates, please scroll down.

Below, there is a menu to choose from different scales of visualization:

* Experiment statistics: it gives informations about the total number of localizations, of clusters as well as the storage size of the directory,
* Boxplots: different scales of data representation can be displayed, for several photophysical parameters (once at a time),
* Plate heatmap: plot a heatmap with a 96-wells plate shape and display the well mean/median of the selected photophysical parameter.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

database = create_database()

# widget creation
w_action = widgets.Dropdown(
    options=['Select a visualization...', 'Experiment Statistics', 'Boxplots', 'Plate Heatmap'],
    value='Select a visualization...',
    description='Action:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

photo_params = ['intensity', 'photon_loc', 'blinks', 'total ON', 'total OFF', 'avg_on', 'avg_off']

w_param = widgets.Dropdown(
    options=photo_params,
    value='intensity',
    description='Parameter:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(display='none', width='400px') 
)

w_grouping = widgets.RadioButtons(
    options=[
        'Compare FOVs (Single well)', 
        'Compare 3 wells (Single protein)'
    ],
    value='Compare 3 wells (Single protein)',
    description='Scale:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(display='none') 
)

protein_list = ['Select a protein...'] + list(database.keys())
w_protein = widgets.Dropdown(
    options=protein_list,
    value=protein_list[0],
    description='Protein:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(display='none', width='400px') 
)

w_well = widgets.Dropdown(
    options=[],
    description='Target Well:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(display='none', width='400px')
)

btn_run = widgets.Button(description='Run', button_style='primary', icon='play')
out = widgets.Output()



def update_ui(*args):
    action = w_action.value
    grouping = w_grouping.value
    protein = w_protein.value
    
    w_param.layout.display = 'none'
    w_grouping.layout.display = 'none'
    w_protein.layout.display = 'none'
    w_well.layout.display = 'none'
    
    if action == 'Plate Heatmap':
        w_param.layout.display = 'block'
        
    elif action == 'Boxplots':
        w_param.layout.display = 'block' 
        w_grouping.layout.display = 'block' 
        w_protein.layout.display = 'block'
        
        # single well
        if grouping == 'Compare FOVs (Single well)':
            w_well.layout.display = 'block'
            if protein != 'Select a protein...':
                w_well.options = database[protein][0] # eg: ['C3', 'D9', 'E3']
            else:
                w_well.options = []

w_action.observe(update_ui, names='value')
w_grouping.observe(update_ui, names='value')
w_protein.observe(update_ui, names='value') 


# main function
def on_run_clicked(b):
    with out:
        clear_output(wait=True)
        
        action = w_action.value
        param = w_param.value
        grouping = w_grouping.value
        protein = w_protein.value
        well = w_well.value
        
        if action == 'Select a visualization...':
            print("Please select a valid action.")
            return
            
        if action == 'Boxplots' and protein == 'Select a protein...':
            print("Error: Please select a specific protein.")
            return
            
        print(f"Launching : {action}...")
        
        if action == 'Experiment Statistics':
            print("Generating global statistics...\n")
            nlocs, nclusters, total_size = count_total_locs_and_clusters(pathway)
            print("Number of localization in HCS-SMLM plate:", nlocs)
            print("Number of clusters in HCS-SMLM plate:", nclusters)
            print("Total storage size of the HCS-SMLM data:", total_size)

            
        elif action == 'Boxplots':
            print(f"Generating Boxplots for parameter: {param.upper()}")
            
            if grouping == 'Compare 3 wells (Single protein)':
                print(f"Comparing 3 wells for {protein}")
                plot_well_boxplots(pathway, param, protein)
                
            elif grouping == 'Compare FOVs (Single well)':
                if not well:
                    print("Error: Please select a target well first.")
                    return
                print(f"Comparing FOVs within well {well} for {protein}")
                plot_fov_boxplots(pathway, param, protein, well)
                
        elif action == 'Plate Heatmap':
            print(f"Generating Plate Heatmap for parameter: {param.upper()}")
            plot_plate_heatmap(pathway, param)

btn_run.on_click(on_run_clicked)


# display
ui = widgets.VBox([
    widgets.HTML("<h3>HCS-SMLM Data Visualization</h3>"),
    w_action,
    w_param,
    w_grouping,
    w_protein, 
    w_well,
    widgets.HTML("<hr>"),
    btn_run,
    out
])

display(ui)

### Export data in CSV/Excel format

It is possible to export the values ​​of each photophysical parameter for each FOV of the same protein in order to perform statistical tests or other visualizations with other software.

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from ipyfilechooser import FileChooser


w_stat = widgets.RadioButtons(
    options=['Median', 'Mean'],
    value='Median',
    description='Statistic:',
    style={'description_width': 'initial'}
)

def create_folder_chooser(index):
    fc = FileChooser('.') 
    fc.show_only_dirs = True 
    fc.title = f'<b>Plate {index} Directory:</b>'
    fc.layout = widgets.Layout(width='600px')
    return fc

path_inputs = [create_folder_chooser(1)]
paths_container = widgets.VBox(path_inputs)

btn_add_path = widgets.Button(description='+ Add Plate', button_style='info')
btn_remove_path = widgets.Button(description='- Remove Plate', button_style='warning')
btn_run = widgets.Button(description='Run Extraction', button_style='primary', icon='play')

out_export = widgets.Output()

def add_path_input(b):
    idx = len(path_inputs) + 1
    new_input = create_folder_chooser(idx)
    path_inputs.append(new_input)
    paths_container.children = path_inputs

def remove_path_input(b):
    if len(path_inputs) > 1:
        path_inputs.pop()
        paths_container.children = path_inputs

def run_extraction(b):
    with out_export:
        clear_output(wait=True)
        selected_paths = [fc.selected_path for fc in path_inputs if fc.selected_path is not None]
        stat_choice = w_stat.value
        
        if len(selected_paths) == 0:
            print("Error : Please select at least one correct plate.")
            return
            
        print(f"Extraction for {len(selected_paths)} plate(s) starting...")
        process_and_export_plates(plate_paths=selected_paths, stat_choice=stat_choice)

btn_add_path.on_click(add_path_input)
btn_remove_path.on_click(remove_path_input)
btn_run.on_click(run_extraction)


box_buttons = widgets.HBox([btn_add_path, btn_remove_path])
ui_export = widgets.VBox([
    widgets.HTML("<h3>HCS-SMLM Excel Exporter</h3>"),
    w_stat,
    widgets.HTML("<b>Select Experiment Folders:</b>"),
    paths_container,
    box_buttons,
    widgets.HTML("<hr>"),
    btn_run,
    out_export
])

display(ui_export)